# Extract CityStatus Data

Extracts CityStatus assets and their per-region attribute effects from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.


In [1]:
from pathlib import Path
import json
import re

import pandas as pd

from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache


## Load assets

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")


Total assets: 30707
Total texts: 32762


## Build the CityStatus -> RequiredPopulation lookup

`EconomyFeature` (one asset) holds `EconomyFeature7.CityStatusFeature.Region.{Roman,Celtic}.CityStatusList`.
Each list item references a CityStatus asset and contains a RequiredPopulation list whose first
entry has the population threshold.


In [3]:
economy_feature = templates["EconomyFeature"].assets[0]
regions = economy_feature.find("EconomyFeature7.CityStatusFeature.Region")

city_status_lists = {
    "Roman": regions.Roman.CityStatusList,
    "Celtic": regions.Celtic.CityStatusList,
}

pop_map: dict[int, int] = {}
for entries in city_status_lists.values():
    for entry in entries:
        city_status = entry.find("CityStatus")
        if city_status is None or city_status() is None:
            continue
        guid = city_status().guid
        required = entry.find("RequiredPopulation")
        pop_count = None
        if required is not None and len(required) > 0:
            pop_count = required[0].find_value("PopulationCount")
        pop_map[guid] = int(pop_count) if pop_count is not None else 0

print(f"Mapped {len(pop_map)} CityStatus -> RequiredPopulation entries")


Mapped 65 CityStatus -> RequiredPopulation entries


## Extract CityStatus rows

In [4]:
from typing import TypedDict

from assetextractor.parsing.core.assets import Asset
from assetextractor.parsing.core.attributes import WandImageProto

ATTRIBUTE_PATHS = {
    "Belief": "Belief.Value",
    "Knowledge": "Knowledge.Value",
    "Prestige": "Prestige.Value",
    "Happiness": "Happiness.Value",
    "Fire Safety": "FireSafety.Value",
    "Health": "Health.Value",
}
EFFECT_CATEGORIES = {
    "Attribute Effects Roman": "CityStatus.AttributeEffectsRoman",
    "Attribute Effects Regional": "CityStatus.AttributeEffectsRegional",
    "Attribute Effects Mixed": "CityStatus.AttributeEffectsMixed",
}


def extract_effects(asset, base_path: str, category: str) -> dict[tuple[str, str], int]:
    out: dict[tuple[str, str], int] = {}
    for attr_label, sub_path in ATTRIBUTE_PATHS.items():
        value = asset.find_value(f"{base_path}.{sub_path}")
        out[(category, attr_label)] = int(value) if value is not None else 0
    return out


class IconData(TypedDict):
    name: str | None
    # url: str | None
    image: WandImageProto | None
    path: str | None

def get_icon_data(asset: Asset) -> IconData:
    """Consolidates icon metadata, URL, image object, and source file path."""
    icon = asset.icon
    
    # 1. Get the name (stem) from the filename if it exists
    name = None
    path = None
    icon_node = asset.find("Standard.IconFilename")
    if icon_node and icon_node.value:
        name = icon_node.value.stem
        path = str(icon_node.value) # The original .dds path

    return {
        "name": name,
        # "url": icon.get_data_url() if icon else None,
        "image": icon.get_image() if icon else None,
        "path": path
    }


rows: list[dict] = []
row_indices: list[str] = []
icons_data: dict[str, IconData] = {}

for asset in templates["CityStatus"].assets:
    name = asset.find_value("Standard.Name")
    row_indices.append(name)

    text = asset.find("CityStatus.CityStatusName")
    english = text().values.get("english") if text and text() is not None else None

    # Consolidated Icon Data for export step
    icon_data = get_icon_data(asset)
    icons_data[asset.guid] = icon_data
    
    row: dict[tuple[str, str], object] = {
        ("UID", ""): asset.guid,
        ("Name", ""): english,
        ("Icon", ""): icon_data["name"],
        ("Required Population", ""): pop_map.get(asset.guid, 0),
    }
    for category, base_path in EFFECT_CATEGORIES.items():
        row.update(extract_effects(asset, base_path, category))
    rows.append(row)
    

df = pd.DataFrame(rows, index=row_indices)
df.columns = pd.MultiIndex.from_tuples(df.columns, names=["Category", "Attribute"])

df_roman = df[df.index.str.contains("Roman", case=False)]
df_celtic = df[df.index.str.contains("Celtic", case=False)]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(f"Extracted {len(df)} CityStatus rows ({len(df_roman)} Roman, {len(df_celtic)} Celtic)")


Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_31_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_32_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_33_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_34_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_35_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_36_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache\data\ui\fhd\base\icon_content\city_status\icon_2d_city_status_37_0.dds does not exist.
Image D:\Anno_117_Modding\asset-extractor\.cache

Extracted 70 CityStatus rows (45 Roman, 25 Celtic)


## Save CSV files

In [5]:
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "city_status.csv")
df_roman.to_csv(output_dir / "city_status_roman.csv")
df_celtic.to_csv(output_dir / "city_status_celtic.csv")

print(f"Saved CSVs to {output_dir.resolve()}")


Saved CSVs to D:\Anno_117_Modding\asset-extractor\assetextractor\conversion\statistics\results\tables


## City Status - Roman

In [6]:
df_roman


Category                 UID                   Name                      Icon  \
Attribute                                                                       
City Status Roman 01    1929                Outpost  icon_2d_city_status_01_0   
City Status Roman 02    3559            Small Vicus  icon_2d_city_status_02_0   
City Status Roman 03    3563            Large Vicus  icon_2d_city_status_03_0   
City Status Roman 04    3564             Minor Town  icon_2d_city_status_04_0   
City Status Roman 05    3573           Growing Town  icon_2d_city_status_05_0   
City Status Roman 06    3579             Large Town  icon_2d_city_status_06_0   
City Status Roman 07    3581             Minor City  icon_2d_city_status_07_0   
City Status Roman 08    3591           Growing City  icon_2d_city_status_08_0   
City Status Roman 09    3595             Major City  icon_2d_city_status_09_0   
City Status Roman 10    3599          Minor Colonia  icon_2d_city_status_10_0   
City Status Roman 11    3606        Growing Colonia  icon_2d_city_status_11_0   
City Status Roman 12    3610          Major Colonia  icon_2d_city_status_12_0   
City Status Roman 13    3611           Vast Colonia  icon_2d_city_status_13_0   
City Status Roman 14    3612    Minor Imperial City  icon_2d_city_status_14_0   
City Status Roman 15    3613  Growing Imperial City  icon_2d_city_status_15_0   
City Status Roman 16    5700    Major Imperial City  icon_2d_city_status_16_0   
City Status Roman 17    5701     Vast Imperial City  icon_2d_city_status_17_0   
City Status Roman 18    5702      Minor Megalopolis  icon_2d_city_status_18_0   
City Status Roman 19   21122    Growing Megalopolis  icon_2d_city_status_19_0   
City Status Roman 20   21123      Major Megalopolis  icon_2d_city_status_20_0   
City Status Roman 21   21124       Vast Megalopolis  icon_2d_city_status_21_0   
City Status Roman 22   21125       Minor Cosmopolis  icon_2d_city_status_22_0   
City Status Roman 23   21126     Growing Cosmopolis  icon_2d_city_status_23_0   
City Status Roman 24   21127       Major Cosmopolis  icon_2d_city_status_24_0   
City Status Roman 25   21128        Vast Cosmopolis  icon_2d_city_status_25_0   
City Status Roman 26   21129     Minor Monstropolis  icon_2d_city_status_26_0   
City Status Roman 27   21130   Growing Monstropolis  icon_2d_city_status_27_0   
City Status Roman 28   21131     Major Monstropolis  icon_2d_city_status_28_0   
City Status Roman 29   21132      Vast Monstropolis  icon_2d_city_status_29_0   
City Status Roman 30   21133              Omnipolis  icon_2d_city_status_30_0   
City Status Roman 31  150661      Growing Omnipolis  icon_2d_city_status_31_0   
City Status Roman 32  150662        Major Omnipolis  icon_2d_city_status_32_0   
City Status Roman 33  150663         Vast Omnipolis  icon_2d_city_status_33_0   
City Status Roman 34  150664     Minor Aureliopolis  icon_2d_city_status_34_0   
City Status Roman 35  150666   Growing Aureliopolis  icon_2d_city_status_35_0   
City Status Roman 36  150667     Major Aureliopolis  icon_2d_city_status_36_0   
City Status Roman 37  150672      Vast Aureliopolis  icon_2d_city_status_37_0   
City Status Roman 38  150673      Minor Titanopolis  icon_2d_city_status_38_0   
City Status Roman 39  150674    Growing Titanopolis  icon_2d_city_status_39_0   
City Status Roman 40  150675      Major Titanopolis  icon_2d_city_status_40_0   
City Status Roman 41  150676       Vast Titanopolis  icon_2d_city_status_41_0   
City Status Roman 42  150677     Minor Pantheopolis  icon_2d_city_status_42_0   
City Status Roman 43  150678   Growing Pantheopolis  icon_2d_city_status_43_0   
City Status Roman 44  150679     Major Pantheopolis  icon_2d_city_status_44_0   
City Status Roman 45  150680      Vast Pantheopolis  icon_2d_city_status_45_0   

Category             Required Population Attribute Effects Roman            \
Attribute                                                 Belief Knowledge   
City Status Roman 01                

## City Status - Celtic

In [7]:
df_celtic


Category                 UID                   Name                      Icon  \
Attribute                                                                       
City Status Celtic 01   8503                Outpost  icon_2d_city_status_01_0   
City Status Celtic 02   8504            Small Vicus  icon_2d_city_status_02_0   
City Status Celtic 03   8505            Large Vicus  icon_2d_city_status_03_0   
City Status Celtic 04   8506             Minor Town  icon_2d_city_status_04_0   
City Status Celtic 05   8507           Growing Town  icon_2d_city_status_05_0   
City Status Celtic 06   8508             Large Town  icon_2d_city_status_06_0   
City Status Celtic 07   8509             Minor City  icon_2d_city_status_07_0   
City Status Celtic 08   8510           Growing City  icon_2d_city_status_08_0   
City Status Celtic 09   8511             Major City  icon_2d_city_status_09_0   
City Status Celtic 10   8512          Minor Colonia  icon_2d_city_status_10_0   
City Status Celtic 11   8513        Growing Colonia  icon_2d_city_status_11_0   
City Status Celtic 12   8514          Major Colonia  icon_2d_city_status_12_0   
City Status Celtic 13   8515           Vast Colonia  icon_2d_city_status_13_0   
City Status Celtic 14   8516    Minor Imperial City  icon_2d_city_status_14_0   
City Status Celtic 15   8517  Growing Imperial City  icon_2d_city_status_15_0   
City Status Celtic 16   8518    Major Imperial City  icon_2d_city_status_16_0   
City Status Celtic 17   8519     Vast Imperial City  icon_2d_city_status_17_0   
City Status Celtic 18   8520      Minor Megalopolis  icon_2d_city_status_18_0   
City Status Celtic 19  21115    Growing Megalopolis  icon_2d_city_status_19_0   
City Status Celtic 20  21116      Major Megalopolis  icon_2d_city_status_20_0   
City Status Celtic 21  21117       Vast Megalopolis  icon_2d_city_status_21_0   
City Status Celtic 22  21118       Minor Cosmopolis  icon_2d_city_status_22_0   
City Status Celtic 23  21119     Growing Cosmopolis  icon_2d_city_status_23_0   
City Status Celtic 24  21120       Major Cosmopolis  icon_2d_city_status_24_0   
City Status Celtic 25  21121        Vast Cosmopolis  icon_2d_city_status_25_0   

Category              Required Population Attribute Effects Roman            \
Attribute                                                  Belief Knowledge   
City Status Celtic 01                   0                       0         0   
City Status Celtic 02                 150                       0         0   
City Status Celtic 03                 350                       1         0   
City Status Celtic 04                 750                       2         1   
City Status Celtic 05                1250                       3         3   
City Status Celtic 06                2500                       4         4   
City Status Celtic 07                3500                       5         6   
City Status Celtic 08                5000                       6         7   
City Status Celtic 09                7500                       7         9   
City Status Celtic 10               10000                       8        10   
City Status Celtic 11               12500                       9        12   
City Status Celtic 12               15000                      10        13   
City Status Celtic 13               17500                      11        15   
City Status Celtic 14               20000                      12        16   
City Status Celtic 15               22500                      13        18   
City Status Celtic 16               25000                      14        19   
City Status Celtic 17               27500                      15        21   
City Status Celtic 18               30000                      16        22   
City Status Celtic 19               32500                      17        24   
City Status Celtic 20               35000                      18        25   
City Status Celtic 21               37500                      19        2

## Export JSON

Nested dict per CityStatus, keyed by a snake_case form of the internal name.


In [8]:
def format_key(name: str) -> str:
    name = name.lower().replace("city status", "")
    return re.sub(r"\s+", "_", name.strip())


JSON_CATEGORY_NAMES = {
    "AttributeEffectsRoman": "Attribute Effects Roman",
    "AttributeEffectsRegional": "Attribute Effects Regional",
    "AttributeEffectsMixed": "Attribute Effects Mixed",
}
JSON_ATTRIBUTE_KEYS = {
    "Belief": "Belief",
    "Knowledge": "Knowledge",
    "Prestige": "Prestige",
    "Happiness": "Happiness",
    "FireSafety": "Fire Safety",
    "Health": "Health",
}


def row_to_dict(row) -> dict:
    icon = row[("Icon", "")]
    out: dict = {
        "UID": int(row[("UID", "")]),
        "Name": str(row[("Name", "")]),
        "Icon": str(icon) if icon is not None else None,
        "RequiredPopulation": int(row[("Required Population", "")]),
    }
    for json_category, df_category in JSON_CATEGORY_NAMES.items():
        out[json_category] = {
            json_attr: int(row[(df_category, df_attr)])
            for json_attr, df_attr in JSON_ATTRIBUTE_KEYS.items()
        }
    return out


def dataframe_to_json(dataframe: pd.DataFrame) -> dict:
    return {format_key(index): row_to_dict(row) for index, row in dataframe.iterrows()}


for name, frame in {
    "city_status": df,
    "city_status_roman": df_roman,
    "city_status_celtic": df_celtic,
}.items():
    target = output_dir / f"{name}.json"
    with target.open("w", encoding="utf-8") as f:
        json.dump(dataframe_to_json(frame), f, indent=4)
    print(f"Wrote {target}")


Wrote results\tables\city_status.json
Wrote results\tables\city_status_roman.json
Wrote results\tables\city_status_celtic.json


## Image Export

Convert every city status image from .DDS to .webp using wand + magick

In [9]:
from wand.image import Image
import os

print("Started Image Export")

for guid, data in icons_data.items():
    img = data["image"]
    original_path = data["path"]
    
    # Skip if there is no image or path associated with this GUID
    if not img or not original_path:
        continue
    
    # Construct output path (swap .dds for .webp)
    output_path = os.path.splitext(original_path)[0] + ".webp"
    
    print(f"Exporting City Status Image - GUID: {guid}")
    print(f"  Source: {os.path.basename(original_path)}")
    print(f"  Target: {output_path}")
    
    try:
        # We use the existing WandImageProto object
        # Ensure format is set to webp for the encoder
        img.format = 'webp'
        img.save(filename=output_path)
    except Exception as e:
        print(f"  [ERROR] Failed to export {guid}: {e}")
    
print("---")
print("Finished Exporting")

Started Image Export
Exporting City Status Image - GUID: 1929
  Source: icon_2d_city_status_01_0.dds
  Target: D:\Anno_117_Modding\asset-extractor\.cache\data\ui\4k\base\icon_content\city_status\icon_2d_city_status_01_0.webp
Exporting City Status Image - GUID: 3559
  Source: icon_2d_city_status_02_0.dds
  Target: D:\Anno_117_Modding\asset-extractor\.cache\data\ui\4k\base\icon_content\city_status\icon_2d_city_status_02_0.webp
Exporting City Status Image - GUID: 3563
  Source: icon_2d_city_status_03_0.dds
  Target: D:\Anno_117_Modding\asset-extractor\.cache\data\ui\4k\base\icon_content\city_status\icon_2d_city_status_03_0.webp
Exporting City Status Image - GUID: 3564
  Source: icon_2d_city_status_04_0.dds
  Target: D:\Anno_117_Modding\asset-extractor\.cache\data\ui\4k\base\icon_content\city_status\icon_2d_city_status_04_0.webp
Exporting City Status Image - GUID: 3573
  Source: icon_2d_city_status_05_0.dds
  Target: D:\Anno_117_Modding\asset-extractor\.cache\data\ui\4k\base\icon_content\c